# BCS quantum simulation: paper reproduction

Notebook reprodutível para acompanhar o manuscrito `papers/bcs_quantum_simulation.md`.

Ele assume que os experimentos já foram executados e que as tabelas/figuras existem em `results/`. A última seção traz os comandos para regenerar tudo.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'results').exists() else cwd.parent
TABLES = ROOT / 'results' / 'tables'
FIGURES = ROOT / 'results' / 'figures'

pd.set_option('display.max_columns', 80)
pd.set_option('display.precision', 6)

## Parâmetros principais

O caso padrão usado no paper é o Hamiltoniano BCS reduzido com quatro modos de pares.

In [ ]:
params = {
    'xi': [-1.0, -0.35, 0.35, 1.0],
    'g': 0.8,
    'occupied': [0, 1],
    'time': 2.5,
    'final_time_observables': 6.0,
    'trotter_steps': 8,
    'trotter_order': 2,
}
params

## Funções auxiliares

In [ ]:
def read_table(name):
    path = TABLES / name
    if not path.exists():
        raise FileNotFoundError(f'Missing table: {path}')
    return pd.read_csv(path)

def show_figure(name, width=850):
    path = FIGURES / name
    if not path.exists():
        raise FileNotFoundError(f'Missing figure: {path}')
    display(Image(filename=str(path), width=width))

def section(title):
    display(Markdown(f'## {title}'))

## Dinâmica BCS e erro de Trotter

In [ ]:
trotter = read_table('bcs_dynamics_trotter.csv')
display(trotter)
best = trotter.sort_values('fidelity', ascending=False).iloc[0]
best[['time', 'steps', 'order', 'fidelity', 'state_error', 'exact_gap_proxy', 'trotter_gap_proxy']]

In [ ]:
show_figure('bcs_trotter_error.png')

## Observáveis físicos para o paper

In [ ]:
obs = read_table('bcs_paper_observables.csv')
final_obs = obs.iloc[-1]
final_obs[['time', 'fidelity', 'state_error', 'exact_pair_number', 'trotter_pair_number', 'exact_gap_proxy', 'trotter_gap_proxy']]

In [ ]:
for fig in [
    'bcs_gap_condensate_time.png',
    'bcs_pair_occupations_time.png',
    'bcs_pair_correlation_exact_final.png',
    'bcs_pair_correlation_trotter_final.png',
]:
    show_figure(fig, width=760)

## Green normal, Gor'kov anômalo e matriz Nambu

In [ ]:
green_spec = read_table('bcs_pair_green_spectral_density.csv')
gorkov_spec = read_table('bcs_gorkov_anomalous_spectrum.csv')
nambu_spec = read_table('bcs_gorkov_nambu_spectral_trace.csv')

summary_spectral = pd.DataFrame([
    {'channel': 'normal G', **green_spec.sort_values('spectral_weight', ascending=False).iloc[0][['omega', 'spectral_weight']].to_dict()},
    {'channel': 'anomalous F', **gorkov_spec.sort_values('spectral_weight', ascending=False).iloc[0][['omega', 'spectral_weight']].to_dict()},
    {'channel': 'Nambu trace', **nambu_spec.sort_values('spectral_weight', ascending=False).iloc[0][['omega', 'spectral_weight']].to_dict()},
])
summary_spectral

In [ ]:
for fig in [
    'bcs_pair_green_time.png',
    'bcs_pair_green_spectrum.png',
    'bcs_gorkov_anomalous_time.png',
    'bcs_gorkov_anomalous_spectrum.png',
    'bcs_gorkov_nambu_channels.png',
    'bcs_gorkov_nambu_spectral_trace.png',
]:
    show_figure(fig, width=820)

## Qiskit shots, energia por Pauli e mitigação

In [ ]:
shots = read_table('bcs_qiskit_shots_noise.csv')
energy = read_table('bcs_qiskit_energy_shots.csv')
mitigation = read_table('bcs_qiskit_error_mitigation.csv')

display(shots[['mode', 'shots', 'total_pair_number', 'total_pair_error']])
display(energy[['mode', 'shots', 'exact_energy', 'statevector_energy', 'energy', 'abs_error_vs_statevector']])
display(mitigation[['mode', 'noise_scale', 'energy', 'abs_error_vs_statevector', 'noise_slope', 'fit_rmse']])

In [ ]:
for fig in [
    'bcs_qiskit_shots_noise_total_pairs.png',
    'bcs_qiskit_shots_noise_occupations.png',
    'bcs_qiskit_energy_shots_values.png',
    'bcs_qiskit_energy_shots_error.png',
    'bcs_qiskit_error_mitigation_zne.png',
    'bcs_qiskit_error_mitigation_errors.png',
]:
    show_figure(fig, width=820)

## Resumo numérico final

In [ ]:
energy_32768 = energy[(energy['mode'] == 'ideal_shots') & (energy['shots'] == 32768)].iloc[0]
noisy_scale_1 = mitigation[(mitigation['mode'] == 'scaled_noise') & (mitigation['noise_scale'] == 1.0)].iloc[0]
zne = mitigation[mitigation['mode'] == 'linear_zne'].iloc[0]

paper_summary = pd.DataFrame([
    {'metric': 'final observable fidelity', 'value': final_obs['fidelity']},
    {'metric': 'final state error', 'value': final_obs['state_error']},
    {'metric': 'normal Green peak omega', 'value': summary_spectral.loc[0, 'omega']},
    {'metric': 'anomalous Green peak omega', 'value': summary_spectral.loc[1, 'omega']},
    {'metric': 'Nambu trace peak omega', 'value': summary_spectral.loc[2, 'omega']},
    {'metric': '32768 ideal shot energy error vs statevector', 'value': energy_32768['abs_error_vs_statevector']},
    {'metric': 'noisy energy error before mitigation', 'value': noisy_scale_1['abs_error_vs_statevector']},
    {'metric': 'energy error after ZNE mitigation', 'value': zne['abs_error_vs_statevector']},
])
paper_summary

## Regenerar resultados

Execute no terminal, a partir da raiz do repositório:

```bash
source .venv/bin/activate
python experiments/run_all.py
pandoc papers/bcs_quantum_simulation.md -s -o papers/bcs_quantum_simulation.html
google-chrome --headless --disable-gpu --no-sandbox \
  --print-to-pdf="$PWD/papers/bcs_quantum_simulation.pdf" \
  "file://$PWD/papers/bcs_quantum_simulation.html"
```

Para validar o núcleo:

```bash
PYTHONPATH=src .venv/bin/python -m pytest -q tests
```